In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "8a5255b0fce49fc730baae6ea6ecc601d45bb16b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/8a5255b0fce49fc730baae6ea6ecc601d45bb16b/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Measuring Classifiers

Training a classifier lowers its cross-entropy loss, but the user of a
classifier asks a different question: how often is its decision right? The
two can disagree. Raising the probability of the correct class of an image
from $0.51$ to $0.99$ changes no decision, since that class had the largest
probability already, yet it lowers the loss from $0.67$ to $0.01$
(that section). The number the user wants is the *accuracy*: the
fraction of examples whose predicted class, the class with the largest
logit, is the true class.

Every classifier in this book therefore reports both numbers on validation
data, the loss because training lowers it and the accuracy because users ask
for it, and most classifiers train with the same optimizer. We write this
code once, in a base class `Classifier` that extends the `Module` class of
that section, so that each model supplies only its network and
its loss. Accuracy has blind spots of its own. When one class is rare, or
when some mistakes cost more than others, it can rank a useless classifier
above a useful one. The last part of the section shows such a case and what
to measure instead.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Loss and Accuracy

A classifier for $q$ classes computes a vector of logits
$\mathbf{o} \in \mathbb{R}^q$ for each example and predicts the class with
the largest logit. Since the softmax preserves the order of the logits, this
is also the class with the largest probability. Let
$y^{(i)} \in \{1, \ldots, q\}$ be the class of example $i$. The accuracy on
$n$ examples is the fraction of examples whose largest logit belongs to the
true class,

$$\textrm{accuracy} = \frac{1}{n} \sum_{i=1}^n \mathbf{1}\Big(\mathop{\mathrm{argmax}}_j o^{(i)}_j = y^{(i)}\Big),$$

where $\mathbf{1}(\cdot)$ is 1 if its argument is true and 0 otherwise. Each
example contributes 1 if its prediction is right and 0 otherwise, and the
accuracy averages these contributions. A classifier that labels 9,000 of the
10,000 Fashion-MNIST test images correctly has accuracy 0.9; its error rate,
one minus the accuracy, is 0.1.

We report the accuracy but never train on it. As that section
showed, whether an example is classified correctly changes in a step when
its *margin*, the logit of the true class minus the largest other logit,
crosses zero. The accuracy is therefore a step function of the parameters,
and its gradient is zero wherever it exists. The cross-entropy loss is a
smooth, decreasing function of the margin, and the sum of the losses divided
by $\log 2$ is an upper bound on the number of training errors.
the figure shows how one vector of logits serves
both purposes.

![One logit vector, two uses. Training (top) turns the logits into probabilities with the softmax and reads off the cross-entropy loss of the true class, which changes smoothly with the parameters. Evaluation (bottom) takes the class with the largest logit and compares it with the label, which counts toward the accuracy and changes in steps. The logits $(1.0, 2.2, 0.3)$ score the classes cat, chicken, and dog for an image of a chicken.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-clf-loss-accuracy.svg)

The two numbers can move in opposite directions. A model that becomes
confidently wrong on a few validation examples pays a large loss on each of
them, so its validation loss can rise while its accuracy still improves. We
therefore report both.

## The `Classifier` Class

Every classification model in the book evaluates itself the same way: on
each validation minibatch it computes the loss and the accuracy. `Classifier`
does this once for all of them. Its `validation_step` replaces that of
`Module`, which returns only the loss, by one that returns a dictionary with
both metrics. The `Trainer` of that section averages each metric
that a step returns over the validation set, weighting every minibatch by
its number of examples. The reported loss and accuracy are therefore exact
means over the validation set, even when the last minibatch is smaller than
the others: with minibatches of 64, the 10,000 Fashion-MNIST test images end
in a minibatch of 16. After each epoch, the `Trainer` appends the two means
to `trainer.history['val_loss']` and `trainer.history['val_acc']`.

The `Trainer` compiles the validation step, which returns both metrics as
arrays; it averages them over the minibatches outside the compiled
function.

In [2]:
class Classifier(d2l.Module):
    """The base class of classification models."""
    def validation_step(self, batch):
        Y_hat = self(*batch[:-1])
        return {'loss': self.loss(Y_hat, batch[-1]),
                'acc': self.accuracy(Y_hat, batch[-1])}

The class leaves the network and the loss to its subclasses: the next
section writes both for softmax regression.

### Computing Accuracy

The method `accuracy` evaluates the equation on a minibatch. `Y_hat`
holds the logits, one row per example (more leading axes are flattened
first). `argmax` along the class axis gives the predicted classes, which we
cast to the data type of the labels `Y`, since some frameworks compare only
values of the same type. The comparison yields 1 for each correct
prediction and 0 for each wrong one, and the mean of these values is the
accuracy. With `averaged=False` the method returns the vector of 0s and 1s
itself, which tells us which examples the model gets wrong.

In [3]:
@d2l.add_to_class(Classifier)
def accuracy(self, Y_hat, Y, averaged=True):
    """Compute the fraction of correct predictions."""
    Y_hat = d2l.reshape(Y_hat, (-1, Y_hat.shape[-1]))
    preds = d2l.astype(d2l.argmax(Y_hat, axis=1), Y.dtype)
    compare = d2l.astype(preds == d2l.reshape(Y, (-1,)), d2l.float32)
    return d2l.reduce_mean(compare) if averaged else compare

The validation step that calls `accuracy` is compiled, so `accuracy` needs no
`jit` of its own.

### The Default Optimizer

Unless a model asks for something else, it trains with minibatch stochastic
gradient descent at its learning rate `self.lr`, as the linear regression of
that section did. The `Trainer` obtains the optimizer once, at
the start of training, by calling the model's method `configure_optimizers`.
We install the default on `Module` rather than on `Classifier`, so that
regression models inherit it as well; later chapters override it to train
with other optimizers.

In [4]:
@d2l.add_to_class(d2l.Module)
def configure_optimizers(self):
    return optax.sgd(self.lr)

## Beyond Accuracy

Accuracy counts every example once and treats every mistake alike. Three
kinds of decision problem break these assumptions: one class is much rarer
than the others, some mistakes cost more than others, and the decision uses
the predicted probabilities themselves rather than only the predicted
class. Each calls for a different measurement.

### Class Imbalance

Consider screening for a disease that affects 1% of the population. A
classifier that ignores its input and always predicts *healthy* is right
for 99% of the people, an accuracy of 0.99, and finds no sick patient. Now
take a screening test that detects 90% of the sick and wrongly flags 5% of
the healthy. Among 100,000 people, 1,000 of them sick, it finds 900 and
misses 100, and it flags 4,950 of the 99,000 healthy people. We count the
four possible outcomes in a table whose rows are the predictions and whose
columns are the true classes:

| | sick | healthy |
|:--|--:|--:|
| **predicted sick** | 900 | 4,950 |
| **predicted healthy** | 100 | 94,050 |

Such a table of counts is a *confusion matrix*. With the sick as the *positive*
class, the four cells are the *true positives* (TP, 900), *false positives*
(FP, 4,950), *false negatives* (FN, 100), and *true negatives* (TN, 94,050).
The test is right for the 94,950 people on the diagonal, an accuracy of
0.95. Accuracy ranks the useless classifier above the useful one. Every
person counts once, so the 1,000 sick people make up only 1% of the
accuracy, however well or badly a classifier treats them.

Two ratios focus on the sick. The *recall* is the fraction of the
sick that the classifier finds, and the *precision* is the fraction of the
flagged people who are sick:

$$\textrm{recall} = \frac{\textrm{TP}}{\textrm{TP} + \textrm{FN}}, \qquad \textrm{precision} = \frac{\textrm{TP}}{\textrm{TP} + \textrm{FP}}.$$

Recall reads the first column of the table, and precision reads its first
row. We compute all three numbers for both classifiers.

In [5]:
def report(name, tp, fp, fn, tn):
    """Print the accuracy, recall, and precision of a 2x2 confusion table."""
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    recall = tp / (tp + fn)                                # Sick found
    precision = tp / (tp + fp) if tp + fp else float('nan')  # Flags right
    print(f'{name}: accuracy {accuracy:.3f}, recall {recall:.3f}, '
          f'precision {precision:.3f}')

report('always healthy', tp=0, fp=0, fn=1000, tn=99000)
report('screening test', tp=900, fp=4950, fn=100, tn=94050)

always healthy: accuracy 0.990, recall 0.000, precision nan
screening test: accuracy 0.950, recall 0.900, precision 0.154


Recall separates the two classifiers: 0 for the one that never flags
anybody, 0.9 for the test. The precision of the always-healthy classifier is
undefined, since it flags nobody. The precision of the test is the
probability that a flagged person is sick, and it is only 0.15. This is the
effect of Bayes' theorem that that section showed
for an HIV test: when a disease is rare, the false positives from the many
healthy people outnumber the true positives, even for an accurate test. A
test with high recall and low precision is still useful for screening, where
every flagged person gets a second, more specific test.

### Asymmetric Costs

Missing a sick patient is worse than a false alarm. Suppose that a false
negative costs 100 times as much as a false positive, $\kappa_\textrm{FN} = 100$
and $\kappa_\textrm{FP} = 1$. The cost of a classifier weights the cells of its
confusion matrix by these costs. Per person, the always-healthy classifier
costs $100 \cdot 1{,}000 / 100{,}000 = 1$ and the screening test
$(100 \cdot 100 + 1 \cdot 4{,}950) / 100{,}000 \approx 0.15$, almost seven
times less. With costs, the useful classifier wins.

Costs also change the decision itself. Suppose a model estimates the
probability $p$ that a patient is sick. Predicting *healthy* costs
$\kappa_\textrm{FN}$ with probability $p$, and predicting *sick* costs
$\kappa_\textrm{FP}$ with probability $1 - p$, so the expected cost is smaller
when we predict *sick* exactly if

$$p \, \kappa_\textrm{FN} > (1 - p)\, \kappa_\textrm{FP}, \quad \textrm{that is,} \quad p > \frac{\kappa_\textrm{FP}}{\kappa_\textrm{FP} + \kappa_\textrm{FN}}.$$

With equal costs the threshold is $\frac{1}{2}$, and for two classes the
rule picks the class with the larger probability, as accuracy assumes. With
$\kappa_\textrm{FN} = 100$ and $\kappa_\textrm{FP} = 1$, the threshold drops to
$1/101 \approx 0.01$: a patient with a 2% chance of being sick should be
flagged. Lowering the threshold raises the recall and lowers the
precision, since more people are flagged. When the costs are unknown,
practitioners often report precision and recall together, or their harmonic
mean, the *F1 score* $2PR/(P + R)$, which is high only when both are.

For $q$ classes the confusion matrix $\mathbf{N}$ has $q \times q$ entries.
With $\hat{y}^{(m)}$ the predicted class of example $m$, the class with the
largest logit,

$$N_{ij} = \sum_{m=1}^n \mathbf{1}\big(\hat{y}^{(m)} = i\big)\, \mathbf{1}\big(y^{(m)} = j\big)$$

counts the examples of class $j$ predicted as class $i$: rows are predicted
classes and columns true classes, as in the table above. Not every library
agrees: scikit-learn's `confusion_matrix` stores the transpose, with the
true classes in the rows, so check the orientation before reading off recall
and precision. The diagonal holds
the correct predictions, so the accuracy is the sum of the diagonal divided
by $n$. Each off-diagonal entry counts one kind of mistake, and costs
$\kappa_{ij}$ for predicting $i$ when the class is $j$ weight these entries
as in the binary case. Dividing each column by its sum gives the matrix
$\mathbf{C}$ with $C_{ij} = N_{ij} / \sum_{i'} N_{i'j}$, the fraction of
the examples of class $j$ that are predicted as class $i$, an estimate of
$P(\hat{y} = i \mid y = j)$. Its diagonal holds the recall of each class.
that section computes these matrices for a classifier
trained on Fashion-MNIST and reads off which garments it confuses.


### Calibration

The rule the equation acts on the probability $p$ itself, so
it gives the cheapest decisions only if the probabilities are right. A
model that reports $p = 0.02$ for a group of patients of whom only 0.2% are
sick flags all of them, although at the threshold of $0.01$ none of them
should be flagged. that section called a classifier *calibrated*
if its probabilities match frequencies, so that of all predictions made
with confidence 0.9, 90% are correct, and noted that networks trained on
cross-entropy tend to be overconfident. The *confidence* of a prediction is
the probability of the predicted class, the largest predicted probability.
A check of the confidence covers only that one probability. The threshold
rule the equation needs $p(\textrm{sick})$ itself to be
calibrated, also when it is small, and we check it the same way, binning the
examples by $p(\textrm{sick})$ and comparing each bin with the fraction of
sick patients in it.

To check calibration on validation data, we sort the examples into bins by
their confidence, for instance $[0.5, 0.7)$, $[0.7, 0.9)$, and $[0.9, 1]$, and
compute in each bin the mean confidence and the accuracy. A *reliability
diagram* plots the accuracy of each bin against its mean confidence. A
calibrated classifier lies on the diagonal; points below it mean that the
classifier is more confident than it is right, or *overconfident*. Here is
such a check for a hypothetical classifier on 10,000 examples:

| confidence bin | examples | mean confidence | accuracy |
|:--|--:|--:|--:|
| $[0.5, 0.7)$ | 1,000 | 0.61 | 0.55 |
| $[0.7, 0.9)$ | 2,000 | 0.81 | 0.70 |
| $[0.9, 1]$ | 7,000 | 0.97 | 0.90 |

In every bin the accuracy falls short of the confidence. The classifier
reports a mean confidence of $0.902$ but is right on only $0.825$ of the
examples. The weighted mean gap between the two columns,
$0.1 \cdot 0.06 + 0.2 \cdot 0.11 + 0.7 \cdot 0.07 = 0.077$, is the *expected
calibration error*, a common one-number summary of the diagram.

Accuracy cannot reveal miscalibration, since it uses only the predicted
class; the reliability diagram can, and it also checks a remedy.
Temperature scaling (that section) divides all logits by one number
$T > 0$, fitted on held-out data, before the softmax. It keeps the order of
the logits, so every prediction and the accuracy stay the same, and with
$T > 1$ it lowers the confidences of an overconfident classifier toward its
accuracy [@Guo.Pleiss.Sun.Weinberger.2017]. A reliability diagram of
the scaled model shows whether the gap has closed.

We can now train a classifier and measure it with more than one number: the
loss that training lowers, the accuracy that users ask about, and, where
classes are rare or mistakes differ in cost, the confusion matrix and the
calibration that accuracy hides. Each of these numbers is measured on a
finite test set, and how far it can be trusted for new data is a question we
return to once a model is trained. The next section writes softmax
regression from scratch as a subclass of `Classifier` and trains it on
Fashion-MNIST.

## Summary

A classifier is trained on its loss but judged by its decisions, so every
classification model in the book reports the cross-entropy loss and the
accuracy, the fraction of examples whose predicted class matches the label.
Accuracy is constant almost everywhere as a function of the parameters and
cannot be trained on directly. The base class `Classifier` returns both
metrics from its validation step, the `Trainer` computes their exact means over
the validation set, and a default `configure_optimizers` supplies minibatch
SGD; a model then provides only its network and its loss. When one class is
rare, accuracy can rank a useless classifier above a useful one; the
confusion matrix, recall, and precision show which mistakes a classifier
makes, costs turn them into a decision threshold, and a reliability diagram
checks whether the predicted probabilities can be trusted.

## Exercises

1. **Optimal decisions and rejection.** In a multiclass problem, let
   $\kappa_{ij}$ be the cost of predicting class $i$ when the true class is
   $j$, and let $p(j \mid x)$ be the probability of class $j$ given the
   input $x$.
    1. Write the expected cost of predicting $i$ in terms of the costs and
       $p(j \mid x)$, and state the rule that selects the optimal $i$.
       Check that it reduces to the equation for two classes.
    1. Let a correct guess cost $0$ and a wrong guess cost $\lambda_s$,
       and allow the classifier to *abstain* instead of guessing, at a
       fixed cost $\lambda_r < \lambda_s$. Derive the threshold on the
       top-class probability above which guessing beats abstaining.
    1. In medical diagnosis, deferring to a specialist is cheap relative to
       a misdiagnosis, say $\lambda_r/\lambda_s = 0.05$. Compute the
       threshold and state what the classifier should do with a prediction
       at $0.9$ confidence. Then describe a case in which reporting the
       full distribution $p(y \mid x)$ serves the physician better than
       either guessing or abstaining.

    *Adapted from Kevin Murphy,
    [Probabilistic Machine Learning: An Introduction](https://probml.github.io/pml-book/book1.html),
    Exercise 5.1.*
1. **Loss at equal accuracy.** Two classifiers $A$ and $B$ both achieve
   90% accuracy on a ten-class test set. On every example that $A$
   classifies correctly, it assigns probability $0.91$ to the correct
   class. On every example that $B$ classifies correctly, it assigns
   $0.51$.
    1. Compute the average cross-entropy loss of each classifier on its
       correctly classified examples. Why would knowing only that $B$'s
       *average* correct-class probability is $0.51$ not have sufficed to
       compute $B$'s loss?
    1. Explain why these numbers do not decide which classifier is safer
       to deploy, and list the further information about the two
       classifiers you would need.
    1. Temperature scaling (that section) divides $B$'s logits
       by $T > 0$ before the softmax and leaves every one of its decisions
       unchanged. Assume
       that on every example the largest logit of $B$ is unique. Show that
       as $T \to 0$ the cross-entropy of $B$ tends to $0$ on the correctly
       classified examples and to $\infty$ on the misclassified ones. What
       does this imply about comparing classifiers by loss alone?
1. **Class-weighted loss.** In the screening example of
   that section, missing a sick patient costs much
   more than a false alarm. Give each class $j$ a weight $w_j > 0$ and
   train $f$ on the class-weighted empirical risk

   $$\frac{1}{n} \sum_{i=1}^n w_{y_i}\, l(f(\mathbf{x}_i), y_i),$$

   where $l$ is the cross-entropy loss and $y_i$ is the class of example
   $i$. This is the weighted empirical risk minimization of
   that section with weights
   $w_{y_i}$ that depend only on the class.
    1. Let the training examples be drawn with class prior $p(y)$ and
       class-conditional densities $p(\mathbf{x} \mid y)$. Show that the
       expected class-weighted loss is proportional to the expected
       unweighted loss under the distribution with the same
       class-conditionals and class prior proportional to $w_y\, p(y)$.
       Name the change to the training data, rather than to the loss,
       that has the same effect.
    1. Fix an input $\mathbf{x}$ with class probabilities
       $p(j \mid \mathbf{x})$, and suppose the model can output any
       distribution $\hat{\mathbf{y}}$. Show that the expected weighted loss
       $$-\sum_j w_j\, p(j \mid \mathbf{x}) \log \hat{y}_j$$
       is minimized by $\hat{y}_j \propto w_j\, p(j \mid \mathbf{x})$.
    1. For the two classes healthy and sick, find the threshold on
       $p(\textrm{sick} \mid \mathbf{x})$ above which the $\arg\max$ of
       this $\hat{\mathbf{y}}$ is sick, and choose the weights so that the
       threshold equals that of the equation. Explain why
       choosing $w_\textrm{sick} > w_\textrm{healthy}$ raises recall, and
       what it does to the number of false positives.
1. **Calibration and usefulness.** A ten-class problem has the same number
   of examples in every class.
    1. A classifier ignores its input and outputs the probability $0.1$ for
       every class, breaking ties at random. Show that it is calibrated,
       and compute its accuracy.
    1. Another classifier is right on 95% of the examples and always
       reports confidence $1$. Draw its reliability diagram and compute its
       expected calibration error.
    1. Explain why calibration and accuracy measure different things, and
       why neither can replace the other when a decision uses the
       threshold the equation.
1. [code] **ROC curves.** A binary classifier that outputs a score
   $s \in (0, 1)$ for the positive class can be thresholded at any
   $\tau \in [0, 1]$, not just $\frac{1}{2}$: predict positive whenever
   $s > \tau$. The *receiver operating characteristic* (ROC) curve plots
   the true-positive rate (recall) against the false-positive rate
   $\textrm{FP}/(\textrm{FP}+\textrm{TN})$ as $\tau$ sweeps from $1$ down
   to $0$. Simulate $1000$ positive and $1000$ negative examples with
   scores $s = \sigma(z)$, where $\sigma(z) = 1/(1 + \exp(-z))$ is the
   sigmoid, $z \sim \mathcal{N}(\mu, 1)$ for positives, and
   $z \sim \mathcal{N}(0, 1)$ for negatives.
    1. Compute and plot the ROC curve for $\mu \in \{0, 1, 2\}$, and state
       which points of the plot the endpoints $\tau = 1$ and $\tau = 0$
       give.
    1. Show that a classifier whose scores are independent of the labels
       traces the diagonal in expectation.
    1. For $\mu = 2$, plot the *precision-recall curve*, precision against
       recall as $\tau$ sweeps, once with equal numbers of positives and
       negatives and once with positives making up 1% of the examples
       (keep 1000 negatives and draw 10 positives, or reweight). Explain why
       the ROC curve does not change with the prevalence while the
       precision-recall curve drops, and which of the two describes the
       screening test of this section better.
1. [code] **AUC as ranking probability.** ● This problem uses the ROC
   curves and simulated scores of problem 5.
    1. Show that, when no two scores tie, the area under the ROC curve
       equals the probability that a randomly chosen positive example
       receives a higher score than a randomly chosen negative one, which
       makes it a threshold-free summary of ranking quality.
    1. Compute this probability in closed form for the simulated scores
       and compare it with the trapezoidal area under each of the curves.

[Discussions](https://d2l.discourse.group/t/17981)